# NeuroTrain Lab — Notebook 1: El Perceptrón

**Tema:** qué es una neurona artificial, para qué sirven las funciones de
activación (ReLU, Sigmoid, Softmax), cómo se apilan en una red multicapa (MLP)
y cómo se ve todo esto como tensores en PyTorch y TensorFlow.

> Primer notebook de 4. Aquí no entrenamos nada todavía: solo entendemos cómo
> una red **predice**. Aprender a que se equivoque menos es el Notebook 2.

## 🎯 Qué aprenderás en este notebook

Al terminar podrás explicar, sin fórmulas de memoria:

1. Qué calcula matemáticamente una neurona (perceptrón).
2. Qué hacen ReLU, Sigmoid y Softmax, y cuándo se usa cada una.
3. Por qué una sola neurona no basta y para qué apilamos capas (MLP).
4. Por qué "forward propagation" es, en el fondo, multiplicar matrices.
5. Qué es un tensor y cómo se ve el mismo cálculo en NumPy, PyTorch y TensorFlow.

**Mapa mental:** `neurona → activación → capa → MLP → forward propagation → tensores`

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import tensorflow as tf
import torch
from sklearn.datasets import make_moons

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data" / "breast_cancer_wisconsin.csv").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from neurotrain.celebrations import celebrate
from neurotrain.visualization import plot_decision_boundary

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)
tf.keras.utils.set_random_seed(RANDOM_STATE)

print("NumPy:", np.__version__, "| PyTorch:", torch.__version__, "| TensorFlow:", tf.__version__)
print("Raíz del proyecto:", PROJECT_ROOT)

## 0. El hilo conductor de los 4 notebooks

En los 4 notebooks vamos a usar el mismo problema real de fondo: **predecir si un
tumor es benigno o maligno** a partir de 30 variables numéricas (dataset *Breast
Cancer Wisconsin*). Aquí solo lo miramos por encima — la auditoría completa y el
entrenamiento real llegan en el Notebook 4.

In [ ]:
DATA_PATH = PROJECT_ROOT / "data" / "breast_cancer_wisconsin.csv"
df = pd.read_csv(DATA_PATH)
print(f"Filas: {df.shape[0]} | Columnas: {df.shape[1]}")
df.head(3)

## 1. Qué es una neurona artificial

<div style="border-left:4px solid #7C3AED; background:#F5F3FF; border-radius:.4rem; padding:.85rem 1.1rem; margin:.7rem 0;">
<b>🧠 CONCEPTO CLAVE — La neurona como un portero de discoteca</b><br><br>
Imagina un portero decidiendo si dejar entrar a alguien. No mira un solo dato:
pondera varios (¿va bien vestido? ¿tiene reserva? ¿es tarde?) y cada criterio
"pesa" distinto para él. Si la suma ponderada supera su umbral de exigencia
(su "actitud" ese día), te deja pasar.

Una neurona artificial hace exactamente eso con números: multiplica cada entrada
`x` por un **weight** que dice cuánto importa, suma un **bias** (lo exigente que
es por defecto) y aplica una función de activación para decidir la salida.
</div>

Matemáticamente, para una neurona con entradas $x_1, x_2, \dots, x_n$:

$$z = w_1 x_1 + w_2 x_2 + \dots + w_n x_n + b$$

$z$ es solo un número. Para convertirlo en una decisión aplicamos una **función de
activación** — eso es la Sección 2. Antes, construyamos $z$ a mano.

In [ ]:
def weighted_sum(x, w, b):
    """z = x·w + b — el cálculo que hace una neurona antes de activar."""
    return np.dot(x, w) + b


# Ejemplo: "¿llevo paraguas?" con 2 entradas: prob. de lluvia, viento (0-1)
x_ejemplo = np.array([0.8, 0.3])
w_ejemplo = np.array([0.9, 0.2])
b_ejemplo = -0.4

z = weighted_sum(x_ejemplo, w_ejemplo, b_ejemplo)
print("z (suma ponderada):", z)

### ✏️ Ejercicio

Completa `weighted_sum_manual` calculando `z` **sin usar `np.dot`**, con un bucle
`for` que recorra `x` y `w` a la vez. Debe dar el mismo resultado que la celda
anterior (`z ≈ 0.98`).

In [ ]:
def weighted_sum_manual(x, w, b):
    total = 0.0
    for xi, wi in zip(x, w):
        total += ✏️✏️✏️
    return total + b

print(weighted_sum_manual(x_ejemplo, w_ejemplo, b_ejemplo))

<details>
<summary><b>Ver solución</b></summary>

```python
def weighted_sum_manual(x, w, b):
    total = 0.0
    for xi, wi in zip(x, w):
        total += xi * wi
    return total + b

print(weighted_sum_manual(x_ejemplo, w_ejemplo, b_ejemplo))
```

</details>

<div style="border-left:4px solid #2563EB; background:#EFF6FF; border-radius:.4rem; padding:.85rem 1.1rem; margin:.7rem 0;">
<b>❓ DUDA PROBABLE — ¿Con qué weight se queda la neurona?</b><br><br>
Con todos. Si una neurona recibe 30 valores (como en nuestro dataset real),
tiene **30 weights** — uno por cada entrada — más un bias. No existe "el weight
de la neurona" en singular; cada conexión de entrada tiene el suyo propio.
</div>

## 2. Funciones de activación: ReLU, Sigmoid y Softmax

<div style="border-left:4px solid #7C3AED; background:#F5F3FF; border-radius:.4rem; padding:.85rem 1.1rem; margin:.7rem 0;">
<b>🧠 CONCEPTO CLAVE — ¿Por qué no dejar z tal cual?</b><br><br>
Si apilamos neuronas sin ninguna activación no lineal entre medias, toda la red
—por muchas capas que tenga— sigue siendo matemáticamente equivalente a **una
sola** transformación lineal. La no linealidad es lo que permite aprender formas
curvas, no solo líneas rectas. Lo comprobamos en la Sección 2.3.
</div>

In [ ]:
def relu(z):
    return np.maximum(0, z)


def sigmoid(z):
    return 1 / (1 + np.exp(-z))


z_valores = np.linspace(-6, 6, 200)

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
axes[0].plot(z_valores, relu(z_valores), color="#7C3AED")
axes[0].set_title("ReLU(z) = max(0, z)")
axes[0].axhline(0, color="#94A3B8", linewidth=0.8)
axes[1].plot(z_valores, sigmoid(z_valores), color="#2563EB")
axes[1].set_title("Sigmoid(z) = 1 / (1 + e⁻ᶻ)")
axes[1].axhline(0.5, color="#94A3B8", linewidth=0.8, linestyle="--")
for axis in axes:
    axis.grid(alpha=0.2)
fig.tight_layout()
plt.show()

print("ReLU(-3) =", relu(-3), "| ReLU(2) =", relu(2))
print("Sigmoid(0) =", sigmoid(0), "| Sigmoid(6) ≈", round(sigmoid(6), 3))

- **ReLU** dice "pasa tal cual si eres positivo, si eres negativo eres cero".
  Se usa casi siempre en las **capas ocultas**: es barata de calcular y ayuda a
  que el gradiente fluya bien (más sobre esto en el Notebook 3).
- **Sigmoid** aplasta cualquier número a un rango (0, 1) — se lee como una
  **probabilidad**. Se usa en la **capa de salida** de clasificación binaria
  (¿maligno o benigno? un único número entre 0 y 1).

### ✏️ Ejercicio

Implementa `sigmoid` y `relu` tú mismo y comprueba que coinciden con Keras.

In [ ]:
def mi_relu(z):
    return np.maximum(✏️✏️✏️, z)


def mi_sigmoid(z):
    return 1 / (1 + np.exp(✏️✏️✏️))


keras_relu = tf.keras.activations.relu(tf.constant([-2.0, 0.0, 3.0])).numpy()
keras_sigmoid = tf.keras.activations.sigmoid(tf.constant([-2.0, 0.0, 3.0])).numpy()

assert np.allclose(mi_relu(np.array([-2.0, 0.0, 3.0])), keras_relu)
assert np.allclose(mi_sigmoid(np.array([-2.0, 0.0, 3.0])), keras_sigmoid)
print("¡Coinciden con Keras!")

<details>
<summary><b>Ver solución</b></summary>

```python
def mi_relu(z):
    return np.maximum(0, z)


def mi_sigmoid(z):
    return 1 / (1 + np.exp(-z))
```

</details>

<div style="border-left:4px solid #F97316; background:#FFF7ED; border-radius:.4rem; padding:.85rem 1.1rem; margin:.7rem 0;">
<b>⚠️ ERROR TÍPICO — La no linealidad importa de verdad</b><br><br>
Comprueba qué pasa si encadenas dos transformaciones **lineales** sin activación
entre medias: sigue siendo una única transformación lineal, sin curvas nuevas.
</div>

In [ ]:
# Dos capas "lineales" (sin activación) compuestas siguen siendo una sola línea
A = np.array([[2.0, 0.0], [0.0, 2.0]])
B = np.array([[1.0, 1.0], [0.0, 1.0]])
compuesta = A @ B
print("Aplicar A y luego B equivale a UNA sola matriz:")
print(compuesta)
print("Por eso metemos ReLU/Sigmoid entre capas: rompen esa equivalencia.")

### Softmax: cuando hay más de 2 clases

Sigmoid da una probabilidad para una sola clase. Cuando hay **varias clases que
se excluyen entre sí** (ej. "manzana / plátano / naranja"), usamos **Softmax**:
convierte una lista de números (*logits*) en probabilidades que **siempre suman 1**.

In [ ]:
def softmax(logits):
    exponentes = np.exp(logits - np.max(logits))  # -max: estabilidad numérica
    return exponentes / exponentes.sum()


logits_fruta = np.array([2.0, 1.0, 0.1])  # puntuación cruda para manzana/plátano/naranja
probabilidades = softmax(logits_fruta)
print("Logits:        ", logits_fruta)
print("Probabilidades:", probabilidades.round(3))
print("Suma:          ", probabilidades.sum())

<div style="text-align:center; opacity:.85; font-style:italic; margin:1.1rem 0; font-size:1.05rem;">🌱 Acabas de sembrar la primera semilla de tu red neuronal.</div>

## 3. De una neurona a un MLP

Una neurona sola solo puede trazar una **línea recta** de separación. Para
aprender formas más complejas, apilamos neuronas en **capas**, y capas en una
**red multicapa (MLP, Multi-Layer Perceptron)**:

`entradas → capa oculta 1 → capa oculta 2 → ... → capa de salida`

El número de neuronas por capa (32, 16, lo que sea) y el número de capas son
**hiperparámetros**: decisiones que tomamos y validamos, no fórmulas que se
deriven del número de entradas.

## 4. Forward propagation como multiplicación de matrices

<div style="border-left:4px solid #7C3AED; background:#F5F3FF; border-radius:.4rem; padding:.85rem 1.1rem; margin:.7rem 0;">
<b>🧠 CONCEPTO CLAVE — Una capa Dense es solo esto</b><br><br>
`Dense(n, activation)` calcula, para **todos los ejemplos del batch a la vez**:

$$H = \text{activación}(X \cdot W + b)$$

$X$ es la matriz de entradas, $W$ la matriz de weights de la capa (una columna
por neurona) y $b$ el vector de biases. Es exactamente la misma cuenta que
hicimos a mano en la Sección 1, aplicada a todas las neuronas de la capa de golpe.
</div>

In [ ]:
# Un MLP de juguete: 2 entradas -> capa oculta de 3 -> salida de 1
x = np.array([[1.0, 2.0]])  # 1 ejemplo, 2 variables

W1 = np.array([[0.5, -0.9, 0.3],
               [0.2,  0.1, -0.1]])   # (2 entradas, 3 neuronas ocultas)
b1 = np.array([0.1, -0.2, 0.05])

z1 = x @ W1 + b1
h1 = relu(z1)
print("z1 (antes de activar):", z1)
print("h1 (tras ReLU):       ", h1, "  <- el -0.9 se convirtió en 0")

W2 = np.array([[0.7], [-0.5], [0.9]])  # (3 entradas, 1 neurona de salida)
b2 = np.array([-0.1])

z2 = h1 @ W2 + b2
y_hat = sigmoid(z2)
print("z2 (antes de activar):", z2)
print("y_hat (tras Sigmoid): ", y_hat, " <- probabilidad final")

### ✏️ Ejercicio

Repite el forward pass anterior para un segundo ejemplo `x2 = [[-1.0, 0.5]]`,
reutilizando los mismos `W1`, `b1`, `W2`, `b2`. Completa la multiplicación de
matrices de la primera capa.

In [ ]:
x2 = np.array([[-1.0, 0.5]])

z1_b = ✏️✏️✏️ + b1
h1_b = relu(z1_b)
z2_b = h1_b @ W2 + b2
y_hat_b = sigmoid(z2_b)
print("y_hat para x2:", y_hat_b)

<details>
<summary><b>Ver solución</b></summary>

```python
x2 = np.array([[-1.0, 0.5]])

z1_b = x2 @ W1 + b1
h1_b = relu(z1_b)
z2_b = h1_b @ W2 + b2
y_hat_b = sigmoid(z2_b)
print("y_hat para x2:", y_hat_b)
```

</details>

## 5. Visualizando por qué hacen falta capas: fronteras de decisión

Con 30 variables reales no podemos "dibujar" la frontera que separa maligno de
benigno. Así que usamos un dataset sintético de 2 variables — `make_moons` —
donde sí podemos verla.

In [ ]:
X_moons, y_moons = make_moons(n_samples=300, noise=0.2, random_state=RANDOM_STATE)

plt.figure(figsize=(4.5, 4))
plt.scatter(X_moons[:, 0], X_moons[:, 1], c=y_moons, cmap="RdBu_r", edgecolor="white")
plt.title("make_moons: 2 clases, frontera curva")
plt.show()

In [ ]:
# Un único perceptrón (equivalente a Dense(1, sigmoid) sin capas ocultas)
perceptron = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(2,)),
    tf.keras.layers.Dense(1, activation="sigmoid"),
])
perceptron.compile(optimizer="adam", loss="binary_crossentropy")
perceptron.fit(X_moons, y_moons, epochs=80, verbose=0)

# Un MLP pequeño con una capa oculta no lineal
mlp = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(2,)),
    tf.keras.layers.Dense(8, activation="relu"),
    tf.keras.layers.Dense(1, activation="sigmoid"),
])
mlp.compile(optimizer="adam", loss="binary_crossentropy")
mlp.fit(X_moons, y_moons, epochs=80, verbose=0)

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
plot_decision_boundary(
    X_moons, y_moons,
    lambda grid: perceptron.predict(grid, verbose=0).ravel(),
    title="Un perceptrón: solo puede trazar una recta", lang="es", ax=axes[0],
)
plot_decision_boundary(
    X_moons, y_moons,
    lambda grid: mlp.predict(grid, verbose=0).ravel(),
    title="MLP (Dense 8, ReLU): curva la frontera", lang="es", ax=axes[1],
)
fig.tight_layout()
plt.show()

<div style="border-left:4px solid #2563EB; background:#EFF6FF; border-radius:.4rem; padding:.85rem 1.1rem; margin:.7rem 0;">
<b>❓ DUDA PROBABLE — ¿Y con el dataset real de 30 variables?</b><br><br>
El mismo principio aplica, solo que la frontera vive en un espacio de 30
dimensiones que no podemos dibujar en papel. `make_moons` existe únicamente para
que **veas** con tus ojos por qué un MLP con activación no lineal separa formas
que un perceptrón solo no puede.

**Reto:** repite esta celda cambiando `make_moons` por
`sklearn.datasets.make_circles(n_samples=300, noise=0.1, factor=0.5, random_state=RANDOM_STATE)`.
</div>

<div style="text-align:center; opacity:.85; font-style:italic; margin:1.1rem 0; font-size:1.05rem;">🚀 La red está tomando forma bajo tus manos.</div>

## 6. Tensores: NumPy, PyTorch y TensorFlow

<div style="border-left:4px solid #7C3AED; background:#F5F3FF; border-radius:.4rem; padding:.85rem 1.1rem; margin:.7rem 0;">
<b>🧠 CONCEPTO CLAVE — Un tensor es solo un array con superpoderes</b><br><br>
Un **tensor** es la misma idea que un array de NumPy (números organizados en
filas/columnas/dimensiones), pero con dos añadidos que nos importarán más
adelante: puede vivir en GPU, y puede **recordar las operaciones que se le
aplicaron** para calcular gradientes automáticamente (lo veremos en el Notebook 2).
Para operaciones básicas de hoy, se comportan igual que un array.
</div>

In [ ]:
# Crear el mismo tensor 2x2 en los tres "idiomas"
datos = [[1.0, 2.0], [3.0, 4.0]]

array_np = np.array(datos)
tensor_torch = torch.tensor(datos)
tensor_tf = tf.constant(datos)

print("NumPy     ->", array_np.shape, array_np.dtype)
print("PyTorch   ->", tensor_torch.shape, tensor_torch.dtype)
print("TensorFlow->", tensor_tf.shape, tensor_tf.dtype)

In [ ]:
# Mismas operaciones básicas en los tres frameworks
print("Suma +10:")
print(" numpy :", array_np + 10)
print(" torch :", (tensor_torch + 10).numpy())
print(" tf    :", (tensor_tf + 10).numpy())

print("\nReshape a (4,):")
print(" numpy :", array_np.reshape(4))
print(" torch :", tensor_torch.reshape(4).numpy())
print(" tf    :", tf.reshape(tensor_tf, (4,)).numpy())

Ahora repetimos **exactamente** el forward pass de la Sección 4, pero calculado en los tres frameworks a la vez — deben dar el mismo número.

In [ ]:
x_np = np.array([[1.0, 2.0]], dtype="float32")

# --- NumPy (lo que ya hicimos) ---
salida_numpy = sigmoid(relu(x_np @ W1 + b1) @ W2 + b2)

# --- PyTorch ---
x_t = torch.tensor(x_np)
W1_t, b1_t = torch.tensor(W1, dtype=torch.float32), torch.tensor(b1, dtype=torch.float32)
W2_t, b2_t = torch.tensor(W2, dtype=torch.float32), torch.tensor(b2, dtype=torch.float32)
salida_torch = torch.sigmoid(torch.relu(x_t @ W1_t + b1_t) @ W2_t + b2_t)

# --- TensorFlow ---
x_tf = tf.constant(x_np)
salida_tf = tf.sigmoid(tf.nn.relu(x_tf @ W1 + b1) @ W2 + b2)

print("NumPy      ->", salida_numpy)
print("PyTorch    ->", salida_torch.numpy())
print("TensorFlow ->", salida_tf.numpy())
print("\n¿Coinciden los tres?", np.allclose(salida_numpy, salida_torch.numpy())
      and np.allclose(salida_numpy, salida_tf.numpy()))

<div style="border-left:4px solid #22C55E; background:#F0FDF4; border-radius:.4rem; padding:.85rem 1.1rem; margin:.7rem 0;">
<b>📌 PARA RECORDAR — No memorices sintaxis de PyTorch</b><br><br>
El resto del proyecto (Notebooks 2-4, la app) usa **TensorFlow/Keras** para
entrenar de verdad. PyTorch aparece aquí — y una vez más en el Notebook 2 — solo
para que reconozcas los mismos conceptos detrás de una sintaxis distinta. No
necesitas dominar PyTorch para seguir el resto del curso.
</div>

### ✏️ Ejercicio

Crea un tensor de PyTorch con los valores `[10, 20, 30, 40, 50, 60]` y dale forma `(2, 3)`.

In [ ]:
valores = torch.tensor([10, 20, 30, 40, 50, 60])
matriz = valores.reshape(✏️✏️✏️)
print(matriz)
print(matriz.shape)

<details>
<summary><b>Ver solución</b></summary>

```python
valores = torch.tensor([10, 20, 30, 40, 50, 60])
matriz = valores.reshape(2, 3)
print(matriz)
print(matriz.shape)
```

</details>

## 🎯 Autoevaluación

Respóndelas sin mirar atrás. No necesitas frases perfectas: explica el mecanismo con tus palabras.

**1. Una neurona recibe 64 valores de entrada. ¿Cuántos weights tiene?**

A. 1, compartido para todas las entradas
B. 64, uno por entrada, más el bias
C. 64, y además otro por cada neurona de la red
D. 0, los weights los tiene la capa, no la neurona

<details>
<summary><b>Ver respuesta</b></summary>

**B.** Cada conexión de entrada tiene su propio weight. 64 entradas → 64 weights + 1 bias.

</details>

**2. ¿Qué produce ReLU cuando la entrada es negativa, por ejemplo -3?**

A. -3 sin cambios
B. 0
C. 3 (el valor absoluto)
D. Un error, ReLU no admite negativos

<details>
<summary><b>Ver respuesta</b></summary>

**B.** ReLU(z) = max(0, z). Cualquier valor negativo se convierte en 0; los positivos pasan igual.

</details>

**3. ¿Por qué el MLP separa `make_moons` y un único perceptrón no puede?**

A. Porque el MLP tiene más datos de entrenamiento
B. Porque el MLP combina varias neuronas con una activación no lineal, permitiendo fronteras curvas
C. Porque el perceptrón usa Sigmoid y el MLP no
D. No hay diferencia real, es cuestión de suerte con la semilla aleatoria

<details>
<summary><b>Ver respuesta</b></summary>

**B.** Un perceptrón solo traza una recta. Apilar neuronas con una no linealidad entre medias permite curvar esa frontera.

</details>

**4. Softmax convierte 3 logits en 3 probabilidades. ¿Qué es siempre cierto del resultado?**

A. Todas valen exactamente 0.33
B. Suman exactamente 1
C. La mayor es siempre mayor que 0.9
D. Pueden ser negativas si el logit lo es

<details>
<summary><b>Ver respuesta</b></summary>

**B.** Softmax normaliza para que las probabilidades de todas las clases sumen 1, sin importar los valores de entrada.

</details>

**5. ¿Qué diferencia hay, para una operación básica como sumar 10, entre un tensor de PyTorch y un array de NumPy?**

A. El resultado numérico es distinto
B. Ninguna en el resultado; el tensor además puede ir a GPU y calcular gradientes
C. Los tensores no admiten reshape
D. Los tensores solo aceptan números enteros

<details>
<summary><b>Ver respuesta</b></summary>

**B.** Para operaciones básicas, el resultado numérico es idéntico. Los tensores añaden capacidades (GPU, autograd) que aprovecharemos en el Notebook 2.

</details>

**6. Explica con tus propias palabras qué hace un forward pass, sin usar la palabra 'magia'.**

<details>
<summary><b>Qué debería incluir una buena respuesta</b></summary>

- Menciona que cada capa hace una multiplicación de matrices más un vector de biases.
- Explica que después de cada capa (menos casi siempre la última) hay una activación no lineal.
- Deja claro que el resultado final es una predicción, no todavía un aprendizaje.

</details>

**7. Un compañero dice: 'con más neuronas en la capa oculta, la red siempre predice mejor'. ¿Estás de acuerdo?**

<details>
<summary><b>Qué debería incluir una buena respuesta</b></summary>

- Distingue entre capacidad (más parámetros) y generalización (predecir bien en datos nuevos).
- Menciona que más neuronas también significa más riesgo de sobreajuste (idea que se retoma en el Notebook 4).
- Concluye que el número de neuronas es un hiperparámetro que se valida, no una ley fija.

</details>

In [ ]:
celebrate(
    "🎉 ¡Enhorabuena! Completaste el Notebook 1: El Perceptrón 🎉",
    "Ya sabes cómo una neurona transforma números en decisiones y cómo se ve eso "
    "como tensores. En el Notebook 2 descubrirás cómo la red aprende de sus errores.",
)